# 💻 Laptop Price Prediction using Multiple Linear Regression
**Course**: Machine Learning / MLOps Lab Assignment  
**Objective**: Predict laptop prices in Indian Rupees (INR / Rs.) based on hardware specifications using Multiple Linear Regression with `scikit-learn`, `pandas`, `numpy`, and `seaborn`.

--- 
## 📌 Step 1: Import Libraries & Generate Synthetic Dataset
Generate a realistic synthetic dataset (`laptop_prices.csv`) with 300 rows containing:
- `RAM_GB` (GB)
- `Storage_SSD_GB` (GB)
- `CPU_Cores` (Count)
- `Screen_Size_Inches` (Inches)
- `Price` (Target in INR / Rs.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set console currency string
CURRENCY_STR = "Rs. "

def generate_synthetic_dataset(filename="laptop_prices.csv", n_samples=300, random_seed=42):
    np.random.seed(random_seed)
    
    ram_options = [4, 8, 16, 32, 64]
    ssd_options = [128, 256, 512, 1024, 2048]
    cpu_cores_options = [4, 6, 8, 12, 16]
    screen_size_options = [13.3, 14.0, 15.6, 16.0, 17.3]
    
    ram = np.random.choice(ram_options, size=n_samples, p=[0.1, 0.3, 0.4, 0.15, 0.05])
    ssd = np.random.choice(ssd_options, size=n_samples, p=[0.1, 0.3, 0.35, 0.2, 0.05])
    cores = np.random.choice(cpu_cores_options, size=n_samples, p=[0.2, 0.3, 0.3, 0.15, 0.05])
    screen_size = np.random.choice(screen_size_options, size=n_samples, p=[0.15, 0.35, 0.35, 0.1, 0.05])
    
    # Price formula with ground truth weights + noise
    base_price = 15000.0
    noise = np.random.normal(loc=0, scale=4000.0, size=n_samples)
    
    price = np.round(
        base_price + (3000.0 * ram) + (55.0 * ssd) + (3500.0 * cores) + (2500.0 * screen_size) + noise, 2
    )
    
    df = pd.DataFrame({
        'RAM_GB': ram,
        'Storage_SSD_GB': ssd,
        'CPU_Cores': cores,
        'Screen_Size_Inches': screen_size,
        'Price': price
    })
    
    df.to_csv(filename, index=False)
    print(f"[SUCCESS] Synthetic dataset generated and saved to '{filename}'.")
    return df

# Generate synthetic dataset
df = generate_synthetic_dataset()

--- 
## 🔍 Step 2: Exploratory Data Analysis (EDA)
Load dataset and inspect dataset dimensions, summary statistics, and column data types using `head()`, `info()`, and `describe()`.

In [ ]:
# Display first 5 rows
print("--- First 5 Rows (df.head()) ---")
display(df.head())

# Dataset info
print("\n--- Dataset Summary (df.info()) ---")
df.info()

# Summary Statistics
print("\n--- Descriptive Statistics (df.describe()) ---")
display(df.describe().T)

--- 
## ✂️ Step 3: Feature & Target Definition & Train-Test Split
Separate the feature matrix ($X$) from the target variable ($y$), and split into **80% Training Data** and **20% Testing Data** (`random_state=42`).

In [ ]:
# Define Features (X) and Target (y)
X = df[['RAM_GB', 'Storage_SSD_GB', 'CPU_Cores', 'Screen_Size_Inches']]
y = df['Price']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Total dataset size : {len(df)} samples")
print(f"Training set size   : {X_train.shape[0]} samples (80%)")
print(f"Testing set size    : {X_test.shape[0]} samples (20%)")

--- 
## 🤖 Step 4: Train Multiple Linear Regression Model
Train `LinearRegression()` on `X_train` and `y_train`, and extract intercept ($b_0$) and feature coefficients ($b_1, b_2, b_3, b_4$).

In [ ]:
# Instantiate and train the model
model = LinearRegression()
model.fit(X_train, y_train)

print("[SUCCESS] Linear Regression Model trained successfully!")
print(f"Intercept (b0): {CURRENCY_STR}{model.intercept_:.2f}")

# Feature Importance Coefficients
coeff_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient (Weight in INR)': model.coef_
})
display(coeff_df)

--- 
## 📊 Step 5: Evaluate Model Performance
Calculate **Mean Absolute Error (MAE)**, **Root Mean Squared Error (RMSE)**, and **$R^2$ Score** on `X_test`.

In [ ]:
# Predict test set prices
y_pred = model.predict(X_test)

# Compute metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE)      : {CURRENCY_STR}{mae:,.2f}")
print(f"Root Mean Squared Error (RMSE) : {CURRENCY_STR}{rmse:,.2f}")
print(f"R-squared (R2) Score           : {r2:.4f} ({r2*100:.2f}% variance explained)")

--- 
## 📈 Step 6: Data Visualization (Actual vs Predicted Prices)
Create a publication-ready scatter plot comparing Actual vs. Predicted Prices with an ideal fit reference line ($y=x$).

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6), dpi=300)

# Scatter plot: Actual vs Predicted
sns.scatterplot(
    x=y_test, 
    y=y_pred, 
    color="#2b5c8f", 
    alpha=0.8, 
    s=70, 
    edgecolor="w", 
    linewidth=0.5,
    label="Test Data Predictions"
)

# Reference diagonal line for perfect prediction
min_val = min(y_test.min(), y_pred.min()) - 3000
max_val = max(y_test.max(), y_pred.max()) + 3000
plt.plot(
    [min_val, max_val], 
    [min_val, max_val], 
    color="#d9534f", 
    linestyle="--", 
    linewidth=2, 
    label="Ideal Perfect Prediction (y = x)"
)

# Text box with evaluation metrics
annotation_text = f"MAE: Rs. {mae:,.2f}\nRMSE: Rs. {rmse:,.2f}\n$R^2$ Score: {r2:.4f}"
plt.gca().text(
    0.05, 0.92, 
    annotation_text, 
    transform=plt.gca().transAxes,
    fontsize=11, 
    verticalalignment='top',
    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.9, edgecolor="#cccccc")
)

plt.title("Actual vs. Predicted Laptop Prices in INR (Multiple Linear Regression)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Actual Price (Rs. / INR)", fontsize=12, labelpad=10)
plt.ylabel("Predicted Price (Rs. / INR)", fontsize=12, labelpad=10)
plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.legend(loc="lower right", frameon=True, facecolor="white")
plt.tight_layout()

# Save plot
output_image = "actual_vs_predicted.png"
plt.savefig(output_image, dpi=300)
plt.show()
print(f"[SUCCESS] Visualization saved successfully as '{output_image}'.")